---
**Script name:** 04_02_02_03_simulation_Huxbear_random_sample.ipynb

**Purpose of script:** This script is to simulate the error in the Huxbear station and to randomly sample a subset of the data to test how the length of the dataframe alters the estimate of the correction value. The next script '04_02_02_04_simulation_Huxbear_seed_04.py should be repeated for each value of random seed that has been selected.

**Author:** Dr Katie Allan

**Date Created:** 2022-07-10

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Change 'path_to_folder' to appropriate folder paths.

---

## Importing libraries that are required

In [1]:
import time
from datetime import timedelta
import pickle
import gc
import pandas as pd
import numpy as np
import os

## Functions that are used throughout the script

In [2]:
def random_subsets(df, rng, sizes):
    #create dictionary to store results
    subsets = {}
    #calculate number of rows in df
    num_rows = len(df)

    #loop to create subset of dataframes
    for size in sizes:
        #set the minimum start value
        min_start_index = 0
        #set the maximum sample value
        max_start_index = num_rows - size + 1
        #randomly select an integer to start the sample from
        start_index = rng.integers(min_start_index, max_start_index)

        #subset the data
        sample = df.iloc[start_index:start_index + size]
        #add the subset to the dictionary
        subsets[f'sample_size_{size}'] = sample.sort_values(by='time', ascending=True)

    return subsets

In [3]:
#Function to resample the time intervals of a dataframe
def resample_time_shifts(df, shift_seconds, total_shifts):
    df.set_index(['time'], inplace=True)
    offset_dfs = []

    for shift in range(total_shifts):
        offset = pd.Timedelta(seconds = shift_seconds * shift)

        resampled = df.resample('30min', origin='epoch', offset=offset)['id'].value_counts().reset_index()
        offset_df = resampled.pivot(index='time', columns='id', values='count').fillna(0)
        offset_df = offset_df.reindex(columns=all_tag_ids, fill_value = 0)

        offset_dfs.append(offset_df)
    
    return offset_dfs

In [4]:
#function to normalize each row of data
def normalize_df_rows(df):
    return df.div(df.sum(axis=1), axis=0)

## Define variables

In [5]:
#set the time and repeats for the script
shift_seconds = 10
total_shifts = 180

In [6]:
#Set the tags to filter by
all_tag_ids = ['55331961','4B551E4B', '78780778', '4C785261', '2D661E19', '33333433', '78341E61', '52616161', '662A6107', '614B2A61', '34332A07', '612A4C1E', '2D2D612A', '2D2A7852', '07071E19', '2D192D78', '1E4B074C', '4B662A33', '19557855', '2A615252', '33663378', '5561331E', '341E4B2A', '1978782D', '2D344B2D', '19616666', '551E3307', '34783378', '66667833', '4B616652', '4C610761', '33331E55', '612A3355', '4B2D664C']

In [ ]:
# File directories
input_dir = Path("../../04_01_data_wrangling/output")
output_dir = Path(".")

## Load Data

In [ ]:
#open pickle file of correct node data
with open(f'{input_dir}/station_data.pickle', 'rb') as file:
    station_dataframes = pickle.load(file)

#Filter anchors between specific time period - keep in line with huxbear_1 simulation
station_dataframes = {key: df.loc[(df['time'] >= '2023-09-20 00:00:00')][['time', 'id']] for key, df in station_dataframes.items()}
station_dataframes = {key: df.loc[(df['time'] <= '2023-11-25 00:00:00')][['time', 'id']] for key, df in station_dataframes.items()}

#simulate the error in huxbear 1 node
first_date = pd.Timestamp('2017-01-01 00:00:00')
huxbear_simulated_time_diff = station_dataframes['CFF730D7207F'].assign(time = station_dataframes['CFF730D7207F']['time'] - (station_dataframes['CFF730D7207F'].iloc[0, 0] - first_date))

C:\Users\katie\AppData\Local\Temp\ipykernel_13440\1978883084.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  station_dataframes = pickle.load(file)


## Data processing and export

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_04.pickle'):
    
    simulation_data_seed_04 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(4),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_04.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_04, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_04 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_04.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_04.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_04, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_05.pickle'):
    
    simulation_data_seed_05 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(5),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_05.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_05, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_05 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_05.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_05.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_05, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_15.pickle'):
    
    simulation_data_seed_15 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(15),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_15.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_15, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_15 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_15.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_15.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_15, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_17.pickle'):
    
    simulation_data_seed_17 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(17),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_17.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_17, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_17 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_17.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_17.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_17, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_24.pickle'):
    
    simulation_data_seed_24 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(24),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_24.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_24, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_24 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_24.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_24.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_24, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_26.pickle'):
    
    simulation_data_seed_26 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(26),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_26.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_26, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_26 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_26.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_26.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_26, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_32.pickle'):
    
    simulation_data_seed_32 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(32),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_32.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_32, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_32 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_32.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_32.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_32, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_33.pickle'):
    
    simulation_data_seed_33 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(33),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_33.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_33, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_33 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_33.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_33.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_33, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_48.pickle'):
    
    simulation_data_seed_48 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(48),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_48.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_48, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_48 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_48.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_48.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_48, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_49.pickle'):
    
    simulation_data_seed_49 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(49),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_49.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_49, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_49 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_49.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_49.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_49, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_50.pickle'):
    
    simulation_data_seed_50 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(50),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_50.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_50, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_50 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_50.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_50.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_50, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_51.pickle'):
    
    simulation_data_seed_51 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(51),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_51.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_51, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_51 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_51.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_51.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_51, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_60.pickle'):
    
    simulation_data_seed_60 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(60),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_60.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_60, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_60 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_60.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_60.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_60, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_61.pickle'):
    
    simulation_data_seed_61 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(61),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_61.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_61, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_61 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_61.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_61.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_61, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_78.pickle'):
    
    simulation_data_seed_78 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(78),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_78.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_78, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_78 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_78.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_78.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_78, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_79.pickle'):
    
    simulation_data_seed_79 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(79),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_79.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_79, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_79 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_79.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_79.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_79, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_82.pickle'):
    
    simulation_data_seed_82 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(82),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_82.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_82, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_82 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_82.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_82.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_82, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_85.pickle'):
    
    simulation_data_seed_85 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(85),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_85.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_85, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_85 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_85.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_85.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_85, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_94.pickle'):
    
    simulation_data_seed_94 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(94),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_94.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_94, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_94 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_94.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_94.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_94, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if not os.path.isfile(f'{output_dir}/normalized_shifts_simulation_data_seed_95.pickle'):
    
    simulation_data_seed_95 = random_subsets(
        huxbear_simulated_time_diff,
        np.random.default_rng(95),
        range(100, 3100, 100)
    )
    
    with open(f'{output_dir}/simulation_data_seed_95.pickle', 'wb') as handle:
        pickle.dump(simulation_data_seed_95, handle, protocol=pickle.HIGHEST_PROTOCOL)

    #Normalize and resample subsets of data
    normalized_shifts_simulation_data_seed_95 = {key: list(map(normalize_df_rows, resample_time_shifts(df, shift_seconds, total_shifts))) for key, df in simulation_data_seed_95.items()}
    
    with open(f'{output_dir}/normalized_shifts_simulation_data_seed_95.pickle', 'wb') as handle:
        pickle.dump(normalized_shifts_simulation_data_seed_95, handle, protocol=pickle.HIGHEST_PROTOCOL)